In [ ]:
# 전체 파이프라인 코드
# 포함 항목: 귀하의 add_features 함수 사용, feature-set 정의, 모델 빌더( LGBM / XGB / RandomForest / Ridge ), Optuna(TPE) + SuccessiveHalvingPruner, 5-fold CV RMSE 반환, 최종 결과 CSV 저장.
# 실행 전: DATA_PATH 를 실제 파일명(예: "train.csv")으로 바꾸세요. 필요하면 N_TRIALS_PER_COMBO 를 줄여 빠르게 탐색한 뒤 늘리세요.

In [ ]:
# 간단한 실행·운영 팁
# 빠른 탐색: 우선 N_TRIALS_PER_COMBO = 20 으로 실행해 유망한 모델/피처셋 선별 → 후보에 대해 trials 늘리기.
# 자원 절약: n_jobs=1 로 설정(특히 Optuna 내부 병렬성 문제를 피하려면)하거나 Optuna를 분산 설정(RDB)으로 사용.
# 최종 검증: Optuna로 찾은 최적 파라미터로 전체 train 데이터에서 재학습 후, 독립된 holdout test(1500개) 에 대해 RMSE 계산해 단일 배포용 성능을 확인하세요.
# 결과 해석: CV RMSE와 Holdout RMSE 차이가 크면 데이터 분포 차이/과적합/데이터 누수를 점검하세요.

In [ ]:
# language: python
# requirements: pandas, numpy, scikit-learn, lightgbm, xgboost, optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
import optuna
from optuna.pruners import SuccessiveHalvingPruner
from optuna.samplers import TPESampler


In [ ]:
# ----------------------------
# 0. 설정 (필요 시 수정)
# ----------------------------
DATA_PATH = "train.csv"   # <-- 실제 CSV 경로로 변경
TARGET = "Calories_Burned"
N_FOLDS = 5
N_TRIALS_PER_COMBO = 40   # 초기 테스트용: 필요시 증가
RANDOM_SEED = 42
N_JOBS = -1               # cross_val_score n_jobs (set to 1 if issues)

In [ ]:
# ----------------------------
# 1. 사용자 제공 add_features 함수 (붙여넣기)
# ----------------------------
def get_activity_intensity_by_age(bpm_series, age_series):
    import numpy as _np, pandas as _pd
    ages = _np.array([15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,95], dtype=float)
    low_cut_points = _np.array([126,124,122,120,117,115,113,111,109,107,105,102,100,98,96,92], dtype=float)
    high_cut_points = _np.array([150,147,145,142,139,137,134,132,129,127,124,122,119,117,114,109], dtype=float)
    age = _pd.to_numeric(age_series, errors="coerce").astype(float).to_numpy()
    bpm = _pd.to_numeric(bpm_series, errors="coerce").astype(float).to_numpy()
    low_cut  = _np.interp(age, ages, low_cut_points)
    high_cut = _np.interp(age, ages, high_cut_points)
    out = _np.where(bpm < low_cut, "Light",
          _np.where(bpm < high_cut, "Moderate", "High"))
    return out

def add_features(df) -> pd.DataFrame:
    import numpy as _np, pandas as _pd
    d = df.copy()
    num_cols = [
        "Exercise_Duration",
        "Body_Temperature(F)",
        "BPM",
        "Height(Feet)",
        "Height(Remainder_Inches)",
        "Weight(lb)",
        "Age",
    ]
    for c in num_cols:
        d[c] = _pd.to_numeric(d[c], errors="coerce")

    d["height_in"] = d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]
    d["height_cm"] = d["height_in"] * 2.54
    d["height_m"] = d["height_cm"] / 100.0
    d["weight_kg"] = d["Weight(lb)"] * 0.45359237
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5 / 9)
    d["bmi"] = d["weight_kg"] / (d["height_m"] ** 2 + 1e-9)

    d["Gender"] = (
        d["Gender"].astype(str).str.strip().str.upper()
        .map({"M": 1, "MALE": 1, "F": 0, "FEMALE": 0})
    )

    d["bmr"] = _np.where(
        d["Gender"] == 1,
        66.47 + (13.75 * d["weight_kg"]) + (5.0 * d["height_cm"]) - (6.76 * d["Age"]),
        _np.where(
            d["Gender"] == 0,
            655.1 + (9.56 * d["weight_kg"]) + (1.85 * d["height_cm"]) - (4.68 * d["Age"]),
            _np.nan
        )
    )

    d["mhr"] = 220 - d["Age"]
    d["activity_intensity"] = get_activity_intensity_by_age(d["BPM"], d["Age"])
    HR_REST = 60
    d["hrr"] = d["mhr"] - HR_REST
    d["hrr_ratio"] = (d["BPM"] - HR_REST) / (d["hrr"] + 1e-9)
    d["thermal_load"] = (d["temp_c"] - 37) * d["Exercise_Duration"]
    d["relative_workload"] = (d["BPM"] * d["weight_kg"] * d["Exercise_Duration"]) / (d["Age"] + 1e-9)
    d["hb_stress"] = d["BPM"] / (d["mhr"] + 1e-9)
    d["metabolic_stress"] = d["BPM"] * d["Exercise_Duration"]
    return d

In [ ]:
# ----------------------------
# 2. 데이터 로드 및 파생 변수 적용
# ----------------------------
df = pd.read_csv(DATA_PATH)
df = add_features(df)

# 카테고리 컬럼
cat_cols = ["Gender", "Weight_Status"]

In [ ]:
# ----------------------------
# 3. Feature-set 정의 (원하시면 조정)
# ----------------------------
raw_num = ["Exercise_Duration", "temp_c", "BPM", "Age", "weight_kg", "height_m"]
feature_sets = {
    "Raw": raw_num,
    "Raw+hr_pct": raw_num + ["BPM", "hrr_ratio"],   # hr_pct ~ hrr_ratio 사용
    "BMI": ["Exercise_Duration", "temp_c", "BPM", "Age", "bmi"],
    "BMI+hr_pct": ["Exercise_Duration", "temp_c", "BPM", "Age", "bmi", "hrr_ratio"],
    "Raw+Derived": raw_num + ["BPM_times_Dur", "Dur_per_kg", "bmi", "thermal_load", "metabolic_stress"]
}

In [ ]:
# ----------------------------
# 4. 모델 빌더들 (Optuna가 trial로 하이퍼파라미터 생성)
# ----------------------------
def lgb_builder(trial):
    return lgb.LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 1500),
        num_leaves=trial.suggest_int("num_leaves", 16, 256),
        learning_rate=trial.suggest_loguniform("learning_rate", 1e-3, 0.2),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        reg_alpha=trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        reg_lambda=trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        random_state=RANDOM_SEED,
        n_jobs=1
    )

def xgb_builder(trial):
    return xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 1500),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_loguniform("learning_rate", 1e-3, 0.2),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_alpha=trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        reg_lambda=trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        random_state=RANDOM_SEED,
        n_jobs=1,
        verbosity=0
    )

def rf_builder(trial):
    return RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000),
        max_depth=trial.suggest_int("max_depth", 5, 50),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        random_state=RANDOM_SEED,
        n_jobs=1
    )

def ridge_builder(trial):
    return Ridge(alpha=trial.suggest_loguniform("alpha", 1e-4, 100.0))

model_builders = {
    "LGBM": lgb_builder,
    "XGB": xgb_builder,
    "RandomForest": rf_builder,
    "Ridge": ridge_builder
}

In [ ]:
# ----------------------------
# 5. 평가(도우미) 함수: RMSE & Optuna objective 생성
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def make_objective(X, y, numeric_cols, cat_cols, model_builder):
    def objective(trial):
        model = model_builder(trial)
        numeric_transformer = Pipeline([("scaler", StandardScaler())])
        preprocessor = ColumnTransformer([
            ("num", numeric_transformer, numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), cat_cols)
        ], remainder="drop")
        pipe = Pipeline([("preproc", preprocessor), ("model", model)])

        # fold별로 학습→평가 → trial.report 로 pruning 가능
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        fold_rmses = []
        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X)):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
            pipe.fit(X_tr, y_tr)
            preds = pipe.predict(X_val)
            fold_rmse = rmse(y_val, preds)
            fold_rmses.append(fold_rmse)
            trial.report(fold_rmse, step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_rmses))
    return objective

In [ ]:
# ----------------------------
# 6. 메인 루프: 모델 × feature-set 마다 Optuna 최적화 수행
# ----------------------------
results = []
for model_name, builder in model_builders.items():
    for fs_name, cols in feature_sets.items():
        print(f"\n=== Tuning {model_name} | Feature: {fs_name} ===")
        # 안전: cols + cat_cols가 df에 모두 있는지 확인
        use_cols = [c for c in cols if c in df.columns] + [c for c in cat_cols if c in df.columns]
        X_sub = df[[c for c in cols if c in df.columns] + [c for c in cat_cols if c in df.columns]]
        y_sub = df[TARGET]

        objective = make_objective(X_sub, y_sub, numeric_cols=[c for c in cols if c in df.columns], cat_cols=[c for c in cat_cols if c in df.columns])
        study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED), pruner=SuccessiveHalvingPruner())
        study.optimize(objective, n_trials=N_TRIALS_PER_COMBO, gc_after_trial=True, show_progress_bar=True)
        best_rmse = float(study.best_value)
        print(f"-> Best CV RMSE: {best_rmse:.5f}")
        results.append({"model": model_name, "feature_set": fs_name, "cv_rmse": best_rmse, "best_params": study.best_params})

In [ ]:
# ----------------------------
# 7. 결과 표로 정리 및 저장
# ----------------------------
res_df = pd.DataFrame(results)
table = res_df.pivot(index="model", columns="feature_set", values="cv_rmse")
table = table[sorted(feature_sets.keys())].round(5)
print("\nFinal RMSE table (best CV RMSE from Optuna):\n")
print(table)
table.to_csv("optuna_model_feature_rmse.csv", index=True)
